# SEM defects: SSD weighted test and matched YOLOS cross-validation on Kaggle

This notebook runs, in order:
1. **SSD weighted final tests**, pitch32 and v2. Each model is trained on all 82 train+val images at its best stage B epoch from the ssl30 cross-validation (pitch32 B_037, v2 B_009), then scored once on the 13 test images.
2. **YOLOS cross-validation on the same setup as SSD and RetinaNet**: 5 grouped folds × pitch32 and v2, weighting power .5 cap 3, 40 stage B epochs, v2 input 320.
3. **YOLOS final tests** for both, at their best CV epochs.
4. **Pitch32 vs v2 paired bootstrap** for YOLOS.
5. **One results file**, `matched_results.zip`, holding `results/results.json`, `results.csv`, all metrics, predictions and drawings.

Nothing needs uploading: the code and all 190 images are downloaded from GitHub.

**Before running, in the right-hand panel (Session options):**
- **Accelerator:** GPU T4 x2 or GPU P100.
- **Internet:** On. This needs a phone-verified Kaggle account.

**How to run:** click **Save Version**, choose **Save & Run All (Commit)**, then **Save**. You can close the browser. Files are kept only this way.

**This needs about 2 sessions.** Kaggle stops a session after 12 hours, so the notebook stops at 11 hours and saves its progress as `state.tar.gz`. When a version finishes:
1. Open its **Output**. If `ALL STEPS FINISHED` is not at the end of the log, more work remains.
2. In the notebook editor, use **Add Input → Your Work → Notebooks** and add **this notebook** (its latest version).
3. **Save Version → Save & Run All** again. Cell 5 restores the saved progress, and finished work is skipped.

Repeat until the log ends with `ALL STEPS FINISHED`, then download `matched_results.zip` from the Output tab.

Kaggle's free GPU quota is 30 hours a week; this work needs roughly 14–18 hours.

## 1. Settings

In [ ]:
import glob, json, os, shutil, subprocess, sys, tarfile, threading, time, zipfile
from pathlib import Path

REPO_URL = 'https://github.com/akshay-raj-m007/MTP_ViT.git'
BRANCH = 'relabel-v2'
FOLDERS = 'configs data experiments/mobilevitv2_ssd scripts sem_yolos'
TIME_LIMIT_HOURS = 11  # stop cleanly before Kaggle's 12-hour limit, so the progress is saved

# Best stage B epochs of SSD weighted from the ssl30 cross-validation (cv5.log, 12 Nov). Fixed before any test run.
SSD_EPOCHS = {'ssd_pitch32_weighted': 37, 'ssd_v2_weighted': 9}
MATCHED = ['yolos_pitch32_weighted_matched', 'yolos_v2_weighted_matched']

BASE = Path('/tmp/mtp')            # code, environments and live runs
ROOT = BASE/'MTP_ViT'
ENVS = BASE/'envs'
WORK = Path('/kaggle/working')     # only this folder is kept after a saved run
STATE = WORK/'state.tar.gz'
LOG = WORK/'run.log'
PY_YOLOS, PY_SSD = ENVS/'yolos/bin/python', ENVS/'ssd/bin/python'
os.environ['SSD_PYTHON'] = str(PY_SSD)   # cv5.py and final_test.py start SSD with this Python
STARTED = globals().get('STARTED', time.time())
BASE.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)


def hours_left():
    return TIME_LIMIT_HOURS-(time.time()-STARTED)/3600


def sh(command, cwd=None, hours=None):
    """Run a shell command, print its output as it comes, and stop it after `hours`. Returns the exit code."""
    process = subprocess.Popen(command, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
    timer = threading.Timer(hours*3600, process.kill) if hours else None
    if timer: timer.start()
    with open(LOG, 'a') as log:
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line)
    code = process.wait()
    if timer: timer.cancel()
    if code: print(f'[exit code {code}] {command[:120]}', flush=True)
    return code


def must(command, cwd=None):
    if sh(command, cwd):
        raise RuntimeError(f'Failed: {command}')

## 2. Get the code and images from GitHub

In [ ]:
if not ROOT.exists():
    must(f'git clone --quiet --depth 1 --branch {BRANCH} --filter=blob:none --sparse {REPO_URL} {ROOT}')
    must(f'git -C {ROOT} sparse-checkout set {FOLDERS}')
COMMIT = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Commit:', COMMIT)
counts = {d: len(list((ROOT/d).iterdir())) for d in ('data/raw/images', 'data/pitch_norm/images')}
print(counts)
assert all(n == 95 for n in counts.values()), 'Expected 95 original and 95 pitch-normalised images'

## 3. Build the two Python environments
About 5 minutes. YOLOS and SSD need different package versions, so each has its own environment.

In [ ]:
TORCH = 'torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121'
UV = f'{sys.executable} -m uv'
must(f'{sys.executable} -m pip install --quiet uv')
for name, requirements in (('yolos', 'requirements-gpu.txt'), ('ssd', 'experiments/mobilevitv2_ssd/requirements.txt')):
    python = ENVS/name/'bin/python'
    if not python.exists():
        must(f'{UV} venv --quiet --python 3.12 --python-preference only-managed {ENVS/name}')
    must(f'{UV} pip install --quiet --python {python} {TORCH}')
    must(f'{UV} pip install --quiet --python {python} -r {ROOT/requirements} setuptools==78.1.0')
    must(f'{python} -c "import torch,sys;print(sys.version.split()[0],torch.__version__,torch.cuda.get_device_name(0));assert torch.cuda.is_available()"')

## 4. Download the pretrained models

In [ ]:
must(f'{PY_YOLOS} -c "from huggingface_hub import snapshot_download as d;print(d(\'hustvl/yolos-tiny\'))"')
must(f'{PY_SSD} experiments/mobilevitv2_ssd/scripts/experiment.py setup', cwd=ROOT)
print('Pretrained models ready.')

## 5. Restore saved progress
Does nothing in the first session. Later, it finds `state.tar.gz` from the previous version added as an input.

In [ ]:
KEEP = ['outputs/cv5', 'outputs/final_runs', 'outputs/mobilevitv2_ssd/final_runs', 'outputs/results']
MODEL_FOLDERS = ('best', 'last', 'stage_A_best', 'stage_B_best')


def snapshot():
    """Save progress to /kaggle/working. Finished CV folds keep their predictions and metrics, not their models."""
    for done in (ROOT/'outputs/cv5/runs').glob('*/DONE'):
        for name in MODEL_FOLDERS:
            shutil.rmtree(done.parent/name, ignore_errors=True)
    partial = WORK/'state.partial.tar.gz'
    with tarfile.open(partial, 'w:gz') as tar:
        for folder in KEEP:
            if (ROOT/folder).exists():
                tar.add(ROOT/folder, arcname=folder)
    partial.replace(STATE)
    print(f'Progress saved: {STATE.stat().st_size/1024**2:.0f} MB, {hours_left():.1f} h left in this session', flush=True)


found = sorted(glob.glob('/kaggle/input/**/state.tar.gz', recursive=True), key=os.path.getmtime)
if found and not (ROOT/'outputs/cv5').exists():
    with tarfile.open(found[-1]) as tar:
        tar.extractall(ROOT)
    shutil.copy(found[-1], STATE)
    print('Restored progress from', found[-1])
else:
    print('Nothing to restore' if not found else 'Already restored')

## 6. Progress check
Safe to run at any time.

In [ ]:
def cv_folder(setup, fold):
    return ROOT/'outputs/cv5/runs'/f'{setup}_fold{fold}'


def final_done(setup):
    run = ROOT/'outputs/mobilevitv2_ssd/final_runs'/setup if setup.startswith('ssd') else ROOT/'outputs/final_runs'/setup/'run'
    return (run/'eval_test/metrics.json').exists()


def progress():
    for setup in SSD_EPOCHS:
        print(f'{setup:32} final test {"done" if final_done(setup) else "to do"}')
    for setup in MATCHED:
        folds = [f for f in range(5) if (cv_folder(setup, f)/'DONE').exists()]
        print(f'{setup:32} CV folds done {folds}   final test {"done" if final_done(setup) else "to do"}')
    print(f'Session time left: {hours_left():.1f} h')


progress()

## 7. SSD weighted final tests (pitch32 and v2)
About 1 hour. Uses the best CV epochs from ssl30 given in cell 1.

In [ ]:
if all(final_done(s) for s in SSD_EPOCHS):
    print('SSD final tests already done')
elif hours_left() > 0.5:
    epochs = ' '.join(f'{s}={e}' for s, e in SSD_EPOCHS.items())
    sh(f'{PY_YOLOS} scripts/final_test.py --setups {" ".join(SSD_EPOCHS)} --epochs {epochs} --device cuda', cwd=ROOT, hours=hours_left()-0.3)
    snapshot()
else:
    print('Not enough time left in this session: continue in the next one')

## 8. YOLOS cross-validation on the SSD/RetinaNet setup
Runs fold by fold and saves after each one. A fold starts only if there is time to finish it; otherwise the next session continues.

In [ ]:
fold_hours = []
stopped = False
must(f'{PY_YOLOS} scripts/cv5.py prepare --setups {" ".join(MATCHED)}', cwd=ROOT)
for fold in range(5):
    for setup in MATCHED:
        if (cv_folder(setup, fold)/'DONE').exists():
            continue
        needed = 1.3*max(fold_hours, default=1.5)
        if hours_left() < needed:
            print(f'Stopping: {hours_left():.1f} h left, a fold needs about {needed:.1f} h. The next session continues.')
            stopped = True; break
        started = time.time()
        sh(f'{PY_YOLOS} scripts/cv5.py train --setups {setup} --folds {fold} --device cuda', cwd=ROOT, hours=hours_left()-0.3)
        if (cv_folder(setup, fold)/'DONE').exists():
            fold_hours.append((time.time()-started)/3600)
        snapshot()
    if stopped:
        break
progress()

## 9. Cross-validation summary
Runs only when all 10 folds are finished.

In [ ]:
CV_DONE = all((cv_folder(s, f)/'DONE').exists() for s in MATCHED for f in range(5))
if CV_DONE:
    must(f'{PY_YOLOS} scripts/cv5.py summary --setups {" ".join(MATCHED)}', cwd=ROOT)
    snapshot()
else:
    print('Cross-validation not finished yet: summary skipped')

## 10. YOLOS final tests at the best CV epochs

In [ ]:
if not CV_DONE:
    print('Cross-validation not finished yet: skipped')
elif all(final_done(s) for s in MATCHED):
    print('YOLOS final tests already done')
elif hours_left() > 0.5:
    sh(f'{PY_YOLOS} scripts/final_test.py --setups {" ".join(MATCHED)} --device cuda', cwd=ROOT, hours=hours_left()-0.3)
    snapshot()
else:
    print('Not enough time left in this session: continue in the next one')

## 11. Pitch32 vs v2 paired bootstrap for YOLOS
About 20–40 minutes on the CPU.

In [ ]:
COMPARE = ROOT/'outputs/cv5/summary'/f'compare_{MATCHED[0]}_vs_{MATCHED[1]}.json'
if not CV_DONE:
    print('Cross-validation not finished yet: skipped')
elif COMPARE.exists():
    print('Comparison already done'); print(COMPARE.read_text())
elif hours_left() > 0.8:
    sh(f'{PY_YOLOS} scripts/compare_cv.py {MATCHED[0]} {MATCHED[1]}', cwd=ROOT, hours=hours_left()-0.2)
    snapshot()
else:
    print('Not enough time left in this session: continue in the next one')

## 12. Results file and download
Writes `matched_results.zip`. It holds `results/results.json` and `results.csv` (settings, CV, test and comparison for every setup run here), plus all metrics, predictions, drawings and the log.

In [ ]:
sh(f'{PY_YOLOS} scripts/collect_results.py', cwd=ROOT)
(ROOT/'outputs/results/kaggle_run.json').write_text(json.dumps(dict(
    commit=COMMIT, ssd_epochs_from_ssl30=SSD_EPOCHS, matched_setups=MATCHED,
    gpu=subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip(),
    note='SSD weighted CV ran on ssl30; its finals and all matched YOLOS work ran on Kaggle (torch 2.5.1).'), indent=2))
skip = set(MODEL_FOLDERS)|{'prepared'}
with zipfile.ZipFile(WORK/'matched_results.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for folder in KEEP:
        for path in (ROOT/folder).rglob('*'):
            parts = set(path.relative_to(ROOT).parts)
            if path.is_file() and not parts & skip and path.suffix in ('.json', '.csv', '.png', '.log', '.txt', ''):
                archive.write(path, path.relative_to(ROOT/'outputs'))
    archive.write(LOG, 'run.log')
snapshot()
print(f"matched_results.zip: {(WORK/'matched_results.zip').stat().st_size/1024**2:.1f} MB")
progress()
finished = all(final_done(s) for s in list(SSD_EPOCHS)+MATCHED) and COMPARE.exists()
print('ALL STEPS FINISHED' if finished else 'NOT FINISHED: add this notebook as an input and Save & Run All again')